In [0]:
from pyspark.sql.functions import col, count, when, lit, round as spark_round, \
                                   sum as spark_sum, explode, split

SILVER_PATH = "/Volumes/workspace/ev-energy/ev-insight/silver/ev_stations/"
GOLD_PATH   = "/Volumes/workspace/ev-energy/ev-insight/gold/"

df = spark.read.format("delta").load(SILVER_PATH)

# mart 1 — network health by region
df.groupBy("state").agg(
    count("*").alias("total_chargers"),
    spark_sum(when(col("status") == "OPERATIONAL", 1).otherwise(0)).alias("operational"),
    spark_sum(when(col("status") == "FAULTED",     1).otherwise(0)).alias("faulted")
).withColumn(
    "fault_rate_pct", spark_round(col("faulted") / col("total_chargers") * 100, 2)
).write.format("delta").mode("overwrite").save(f"{GOLD_PATH}mart_network_health/")
print("mart_network_health written")

# mart 2 — ubitricity vs competitors
df.withColumn(
    "operator_type",
    when(col("operator").contains("ubitricity"), lit("Ubitricity")).otherwise(lit("Competitor"))
).groupBy("operator", "operator_type", "state").agg(
    count("*").alias("total_chargers")
).orderBy(col("total_chargers").desc()) \
 .write.format("delta").mode("overwrite").save(f"{GOLD_PATH}mart_competitor_analysis/")
print("mart_competitor_analysis written")

# mart 3 — connector type breakdown
df.select("state", explode(split(col("connector_types"), r"\|")).alias("connector_type")) \
  .groupBy("connector_type", "state").agg(count("*").alias("charger_count")) \
  .orderBy(col("charger_count").desc()) \
  .write.format("delta").mode("overwrite").save(f"{GOLD_PATH}mart_connector_distribution/")
print("mart_connector_distribution written")

# mart 4 — coverage gap (underserved towns first)
df.groupBy("state", "town").agg(
    count("*").alias("total_chargers"),
    spark_sum(when(col("status") == "OPERATIONAL", 1).otherwise(0)).alias("active_chargers")
).withColumn(
    "coverage_score", spark_round(col("active_chargers") / col("total_chargers") * 100, 2)
).orderBy(col("total_chargers").asc()) \
 .write.format("delta").mode("overwrite").save(f"{GOLD_PATH}mart_coverage_gap/")
print("mart_coverage_gap written")